# SAE + SINDy

Caso SAE: os vetores latentes do autoencoder são usados como coordenadas reduzidas `z_i(t)` para ajuste SINDy.

Este notebook é autocontido: não importa `pod_sindy_analysis.py`/`sae_sindy_analysis.py`.

A estrutura está separada por etapas: configuração, carregamento dos dados, biblioteca SINDy, identificação, simulação, visualização/reconstrução e execução.


## 1. Importações e caminhos do caso


In [ ]:
# Esta célula importa bibliotecas numéricas, visualização, regressão esparsa e integração temporal usadas no notebook.
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib
# Força backend não-interativo para evitar conflito matplotlib/matplotlib_inline no Jupyter.
# Este erro aparece como: AttributeError: 'RcParams' object has no attribute '_get'.
matplotlib.use("Agg", force=True)
matplotlib.rcParams["backend"] = "Agg"

import matplotlib.pyplot as plt
plt.switch_backend("Agg")
import numpy as np
from scipy.integrate import solve_ivp
from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)


# Função auxiliar de robustez: força backend Agg antes de criar figuras.
# Evita conflito entre matplotlib antigo do sistema e matplotlib_inline novo do Jupyter.
def force_matplotlib_agg() -> None:
    matplotlib.use("Agg", force=True)
    matplotlib.rcParams["backend"] = "Agg"
    try:
        plt.switch_backend("Agg")
    except Exception:
        pass


In [ ]:
# Esta célula define os caminhos do diretório atual, do caso OpenFOAM e da pasta de resultados SINDy.
SRC_DIR = Path(r"/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/src").resolve()
CASE_DIR = SRC_DIR.parent.parent
OUTPUT_ROOT = CASE_DIR / "SINDy" / "resultados"

print(f"SRC_DIR     = {SRC_DIR}")
print(f"CASE_DIR    = {CASE_DIR}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")


## 2. Carregamento e organização dos dados reduzidos
Define as estruturas de dados e as rotinas para localizar, ordenar e validar as coordenadas reduzidas.


In [ ]:
# Esta célula define estruturas de dados e funções para ler e preparar as coordenadas reduzidas.
# Classe de dados: guarda nome do caso reduzido, símbolo das variáveis, estados reduzidos, tempos, arquivo-fonte e metadados.
@dataclass
class ReducedDataset:
    name: str
    state_symbol: str
    states: np.ndarray
    times: np.ndarray
    source: Path
    metadata: dict[str, Any] = field(default_factory=dict)


# Classe de configuração: concentra os parâmetros do ajuste SINDy e das figuras para facilitar mudanças em uma única célula.
@dataclass
class SindyConfig:
    polynomial_order: int = 2
    use_sine: bool = True
    alpha_values: tuple[float, ...] = (1e-3, 1e-2, 1e-1)
    max_plot_states: int = 6
    heatmap_times: tuple[float, ...] = ()
    heatmap_grid_size: int = 64



# Função de erro: calcula o erro relativo L2 entre a referência e a estimativa.
def relative_l2(reference: np.ndarray, estimate: np.ndarray) -> float:
    return float(np.linalg.norm(reference - estimate) / (np.linalg.norm(reference) + 1e-12))


# Função auxiliar: converte objetos NumPy/listas/dicionários para tipos serializáveis em JSON.
def make_json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: make_json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


# Função auxiliar: ordena os snapshots pelo tempo e remove tempos duplicados mantendo a primeira ocorrência.
def sort_and_deduplicate(times: np.ndarray, states: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(times, kind="stable")
    times = np.asarray(times[order], dtype=np.float64)
    states = np.asarray(states[order], dtype=np.float64)
    unique_times, unique_idx = np.unique(times, return_index=True)
    if len(unique_times) != len(times):
        times = times[unique_idx]
        states = states[unique_idx]
    return times, states


# Função auxiliar: estima o passo temporal típico usando a mediana de diff(times).
def infer_dt(times: np.ndarray) -> float:
    deltas = np.diff(times)
    return 0.0 if deltas.size == 0 else float(np.median(deltas))


# Função de carregamento SAE: lê latents.npz, junta treino/validação quando necessário e organiza os vetores latentes.
def load_sae_dataset(case_dir: Path) -> ReducedDataset:
    # Procura os latentes SAE no caminho real e também no caminho usado por versões antigas.
    latent_candidates = (
        Path(case_dir) / "SAE" / "resultados" / "latents.npz",
        Path(case_dir) / "SAE" / "resultados" / "SAE" / "latents.npz",
    )
    latent_path = next((path for path in latent_candidates if path.exists()), None)
    if latent_path is None:
        checked = "\n".join(str(path) for path in latent_candidates)
        raise FileNotFoundError(f"SAE latent results not found. Checked:\n{checked}")

    data = np.load(latent_path, allow_pickle=True)
    if {"z_train", "z_val", "t_train", "t_val"}.issubset(set(data.files)):
        states = np.vstack(
            [
                np.asarray(data["z_train"], dtype=np.float64),
                np.asarray(data["z_val"], dtype=np.float64),
            ]
        )
        times = np.concatenate(
            [
                np.asarray(data["t_train"], dtype=np.float64),
                np.asarray(data["t_val"], dtype=np.float64),
            ]
        )
    elif "latent_data" in data.files:
        states = np.asarray(data["latent_data"], dtype=np.float64)
        times = (
            np.asarray(data["times"], dtype=np.float64)
            if "times" in data.files
            else np.arange(states.shape[0], dtype=np.float64)
        )
    else:
        raise KeyError("Unsupported SAE latent file format.")

    times, states = sort_and_deduplicate(times, states)
    return ReducedDataset(
        name="sae",
        state_symbol="z",
        states=states,
        times=times,
        source=latent_path,
        metadata={"latent_dim": int(states.shape[1]), "dt": infer_dt(times)},
    )


## 3. Parâmetros do experimento
Ajuste esta célula antes de executar o fluxo completo.


In [ ]:
# Esta célula concentra os parâmetros que controlam a biblioteca, regularização, número de modos/estados e heatmaps.
# Parâmetros principais
config = SindyConfig(
    polynomial_order=2,
    use_sine=True,
    alpha_values=(1e-3, 1e-2, 1e-1),
    max_plot_states=6,
    heatmap_times=(),          # exemplo: (1.0, 2.0, 3.0)
    heatmap_grid_size=64,
)


## 4. Biblioteca SINDy, derivadas, ajuste esparso e simulação
Monta a biblioteca polinomial/trigonométrica, calcula derivadas temporais, ajusta Lasso e integra o sistema identificado.


In [ ]:
# Esta célula implementa o núcleo SINDy: biblioteca Theta, derivadas, regressão esparsa, equações e simulação.
# Função da biblioteca SINDy: gera todas as combinações de potências polinomiais até a ordem escolhida.
def generate_powers(n_variables: int, order: int) -> list[tuple[int, ...]]:
    if n_variables == 1:
        return [(power,) for power in range(order + 1)]
    powers: list[tuple[int, ...]] = []
    for power in range(order + 1):
        for suffix in generate_powers(n_variables - 1, order - power):
            powers.append((power,) + suffix)
    return sorted(powers, reverse=False)


# Classe da biblioteca SINDy: constrói e aplica a matriz Theta(X) com termos polinomiais e, opcionalmente, seno/cosseno.
@dataclass
class PolynomialLibrary:
    n_variables: int
    order: int
    use_sine: bool
    variable_symbol: str

    # Método de inicialização: pré-calcula as potências dos monômios e os nomes das features da biblioteca.
    def __post_init__(self) -> None:
        self.powers = generate_powers(self.n_variables, self.order)
        self.feature_names = self._build_feature_names()

    # Método auxiliar: cria nomes legíveis para cada termo da biblioteca, como 1, a1, a1*a2, sin(a1).
    def _build_feature_names(self) -> list[str]:
        names = []
        for power in self.powers:
            if sum(power) == 0:
                names.append("1")
                continue
            parts = []
            for index, exponent in enumerate(power):
                if exponent == 0:
                    continue
                variable = f"{self.variable_symbol}{index + 1}"
                parts.append(variable if exponent == 1 else f"{variable}^{exponent}")
            names.append("*".join(parts))
        if self.use_sine:
            for index in range(self.n_variables):
                variable = f"{self.variable_symbol}{index + 1}"
                names.append(f"sin({variable})")
                names.append(f"cos({variable})")
        return names

    # Método de transformação: avalia a biblioteca Theta para todos os estados reduzidos.
    def transform(self, states: np.ndarray) -> np.ndarray:
        states = np.asarray(states, dtype=np.float64)
        theta = np.empty((states.shape[0], len(self.feature_names)), dtype=np.float64)
        for column, power in enumerate(self.powers):
            theta[:, column] = np.prod(np.power(states, power), axis=1)
        next_column = len(self.powers)
        if self.use_sine:
            for index in range(self.n_variables):
                theta[:, next_column] = np.sin(states[:, index])
                theta[:, next_column + 1] = np.cos(states[:, index])
                next_column += 2
        return theta

    # Método de transformação unitária: avalia Theta para apenas um estado, usado dentro da integração temporal.
    def transform_single(self, state: np.ndarray) -> np.ndarray:
        return self.transform(np.asarray(state, dtype=np.float64)[None, :])[0]


# Função de derivadas: calcula dX/dt numericamente a partir das coordenadas reduzidas e dos tempos.
def compute_derivatives(states: np.ndarray, times: np.ndarray) -> np.ndarray:
    return np.gradient(states, times, axis=0, edge_order=2 if len(times) > 2 else 1)


# Função de regressão esparsa: ajusta uma equação SINDy com Lasso para uma coordenada/derivada.
def fit_sparse_model(theta: np.ndarray, target: np.ndarray, alpha: float) -> np.ndarray:
    scale = np.linalg.norm(theta, axis=0)
    scale[scale < 1e-12] = 1.0
    theta_scaled = theta / scale[None, :]
    model = Lasso(alpha=alpha, fit_intercept=False, max_iter=20000, tol=1e-5)
    model.fit(theta_scaled, target)
    return model.coef_ / scale


# Função principal SINDy: testa valores de alpha, descobre os coeficientes Xi das equações e escolhe o melhor ajuste de derivadas.
def discover_equations(theta: np.ndarray, derivatives: np.ndarray, alpha_values: tuple[float, ...]):
    print(f"   Library size: {theta.shape[1]} features")
    print(f"   Testing {len(alpha_values)} alpha values")

    # Número de equações a identificar: uma equação diferencial para cada coordenada reduzida.
    n_states = derivatives.shape[1]
    results = []
    # Para cada alpha, ajusta uma matriz Xi diferente e mede erro/esparsidade.
    for alpha in alpha_values:
        # Xi tem dimensão: número de termos da biblioteca x número de coordenadas.
        coefficients = np.zeros((theta.shape[1], n_states), dtype=np.float64)
        # Ajusta separadamente d(state_i)/dt = Theta @ Xi_i.
        for index in range(n_states):
            coefficients[:, index] = fit_sparse_model(theta, derivatives[:, index], alpha)
        # Depois de ajustar Xi, calcula as derivadas previstas pelo modelo.
        derivatives_pred = theta @ coefficients
        result = {
            "alpha": float(alpha),
            "Xi": coefficients,
            "derivatives_pred": derivatives_pred,
            "derivative_l2_error": relative_l2(derivatives, derivatives_pred),
            "n_nonzero": int(np.count_nonzero(np.abs(coefficients) > 1e-8)),
        }
        print(
            f"   alpha={alpha:.4g} | derivative L2={result['derivative_l2_error']:.6f} "
            f"| nonzero terms={result['n_nonzero']}"
        )
        results.append(result)

    # Escolhe o alpha que melhor reproduz as derivadas numéricas.
    best_result = min(results, key=lambda item: item["derivative_l2_error"])
    print(
        f"   Selected alpha={best_result['alpha']:.4g} "
        f"with derivative L2={best_result['derivative_l2_error']:.6f}"
    )
    return best_result, results


# Função de simulação: integra no tempo o sistema dinâmico identificado pelo SINDy usando solve_ivp.
def simulate_sindy(
    library: PolynomialLibrary,
    coefficients: np.ndarray,
    times: np.ndarray,
    initial_state: np.ndarray,
) -> tuple[np.ndarray | None, str | None]:
    # Função interna do integrador: calcula dX/dt = Theta(X) @ Xi no estado atual.
    def rhs(_time: float, state: np.ndarray) -> np.ndarray:
        return library.transform_single(state) @ coefficients

    solution = solve_ivp(
        rhs,
        (float(times[0]), float(times[-1])),
        np.asarray(initial_state, dtype=np.float64),
        t_eval=np.asarray(times, dtype=np.float64),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    if not solution.success or solution.y.shape[1] != len(times):
        return None, str(solution.message)
    return solution.y.T, None


# Função de fallback: faz previsão de um passo usando o estado de referência anterior, útil quando a integração livre falha.
def simulate_reference_guided_sindy(
    library: PolynomialLibrary,
    coefficients: np.ndarray,
    times: np.ndarray,
    reference_states: np.ndarray,
) -> np.ndarray:
    predicted = np.zeros_like(reference_states, dtype=np.float64)
    predicted[0] = reference_states[0]
    for index in range(1, len(times)):
        dt = float(times[index] - times[index - 1])
        rhs = library.transform_single(reference_states[index - 1]) @ coefficients
        predicted[index] = reference_states[index - 1] + dt * rhs
    return predicted


# Função de saída textual: transforma a matriz Xi em equações legíveis para cada coordenada reduzida.
def format_equations(coefficients: np.ndarray, feature_names: list[str], state_symbol: str) -> str:
    lines = []
    for state_index in range(coefficients.shape[1]):
        parts = []
        for value, name in zip(coefficients[:, state_index], feature_names):
            if abs(value) <= 1e-8:
                continue
            parts.append(f"{value:+.6e}" if name == "1" else f"{value:+.6e}*{name}")
        lines.append(f"d{state_symbol}{state_index + 1}/dt = {' '.join(parts) if parts else '0.0'}")
    return "\n".join(lines)


## 5. Gráficos diagnósticos no espaço reduzido
Funções para varredura de `alpha`, comparação de derivadas, trajetória e mapa temporal das coordenadas reduzidas.


In [ ]:
# Esta célula contém funções que geram figuras diagnósticas no espaço reduzido.
# Função de gráfico: mostra como o erro das derivadas e o número de termos não nulos variam com alpha.
def plot_alpha_scan(scan_results: list[dict[str, Any]], output_dir: Path, title: str) -> None:
    force_matplotlib_agg()
    alphas = np.array([item["alpha"] for item in scan_results], dtype=np.float64)
    deriv_errors = np.array([item["derivative_l2_error"] for item in scan_results], dtype=np.float64)
    nonzero = np.array([item["n_nonzero"] for item in scan_results], dtype=np.float64)

    fig, axis_left = plt.subplots(figsize=(8, 4.5))
    axis_left.plot(alphas, deriv_errors, marker="o", linewidth=2, color="tab:blue")
    axis_left.set_xscale("log")
    axis_left.set_xlabel("alpha")
    axis_left.set_ylabel("Derivative relative L2", color="tab:blue")
    axis_left.tick_params(axis="y", labelcolor="tab:blue")
    axis_left.grid(True, alpha=0.3)

    axis_right = axis_left.twinx()
    axis_right.plot(alphas, nonzero, marker="s", linewidth=2, color="tab:red")
    axis_right.set_ylabel("Nonzero terms", color="tab:red")
    axis_right.tick_params(axis="y", labelcolor="tab:red")

    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_dir / "alpha_scan.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de gráfico: compara a trajetória reduzida de referência com a trajetória prevista pelo SINDy.
def plot_state_comparison(
    times: np.ndarray,
    states: np.ndarray,
    states_pred: np.ndarray | None,
    label_reference: str,
    label_model: str,
    output_dir: Path,
    state_symbol: str,
    max_states: int,
) -> None:
    force_matplotlib_agg()
    n_states = min(max_states, states.shape[1])
    fig, axes = plt.subplots(n_states, 1, figsize=(12, 3 * n_states), sharex=True)
    if n_states == 1:
        axes = [axes]
    for index in range(n_states):
        axes[index].plot(times, states[:, index], linewidth=2, color="tab:blue", label=label_reference)
        if states_pred is not None:
            axes[index].plot(
                times,
                states_pred[:, index],
                linestyle="--",
                linewidth=2,
                color="tab:orange",
                label=label_model,
            )
        axes[index].set_ylabel(f"{state_symbol}{index + 1}")
        axes[index].grid(True, alpha=0.3)
        axes[index].legend(loc="upper right")
    axes[-1].set_xlabel("Time")
    fig.suptitle(f"{label_model} trajectory vs {label_reference}")
    fig.tight_layout()
    plt.savefig(output_dir / "trajectory_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de gráfico: compara derivadas numéricas de referência com derivadas previstas pela biblioteca SINDy.
def plot_derivative_comparison(
    times: np.ndarray,
    derivatives: np.ndarray,
    derivatives_pred: np.ndarray,
    output_dir: Path,
    state_symbol: str,
    max_states: int,
) -> None:
    force_matplotlib_agg()
    n_states = min(max_states, derivatives.shape[1])
    fig, axes = plt.subplots(n_states, 1, figsize=(12, 3 * n_states), sharex=True)
    if n_states == 1:
        axes = [axes]
    for index in range(n_states):
        axes[index].plot(times, derivatives[:, index], linewidth=2, color="tab:green", label="Reference")
        axes[index].plot(
            times,
            derivatives_pred[:, index],
            linestyle="--",
            linewidth=2,
            color="tab:red",
            label="SINDy",
        )
        axes[index].set_ylabel(f"d{state_symbol}{index + 1}/dt")
        axes[index].grid(True, alpha=0.3)
        axes[index].legend(loc="upper right")
    axes[-1].set_xlabel("Time")
    fig.suptitle("Derivative comparison")
    fig.tight_layout()
    plt.savefig(output_dir / "derivative_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de gráfico: gera mapa temporal das amplitudes das coordenadas reduzidas.
def plot_state_heatmap(
    times: np.ndarray,
    states: np.ndarray,
    output_dir: Path,
    state_symbol: str,
    title: str,
) -> None:
    force_matplotlib_agg()
    fig, axis = plt.subplots(figsize=(12, 4.5))
    image = axis.imshow(
        states.T,
        aspect="auto",
        origin="lower",
        extent=[times[0], times[-1], 1, states.shape[1]],
        cmap="viridis",
    )
    axis.set_xlabel("Time")
    axis.set_ylabel(f"{state_symbol} index")
    axis.set_title(title)
    fig.colorbar(image, ax=axis, label="Amplitude")
    plt.tight_layout()
    plt.savefig(output_dir / "reduced_state_heatmap.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


## 6. Reconstrução e heatmaps em tempos solicitados
Rotinas opcionais para reconstruir campos e comparar referência x SINDy em tempos específicos.


In [ ]:
# Esta célula contém funções para reconstrução física dos campos e heatmaps em tempos específicos.
# Função auxiliar de heatmaps: associa tempos solicitados aos snapshots disponíveis dentro de uma tolerância.
def match_requested_times(
    times: np.ndarray,
    target_times: tuple[float, ...],
) -> tuple[list[dict[str, float]], list[float], float]:
    dt = infer_dt(times)
    tolerance = max(1e-8, 0.51 * abs(dt) if dt > 0 else 1e-8)
    matched: list[dict[str, float]] = []
    missing: list[float] = []
    for target in target_times:
        index = int(np.argmin(np.abs(times - target)))
        actual = float(times[index])
        if abs(actual - target) <= tolerance:
            matched.append({"requested_time": float(target), "actual_time": actual, "index": index})
        else:
            missing.append(float(target))
    return matched, missing, tolerance


# Função auxiliar SAE: tenta inferir uma grade quadrada quando o vetor já representa uma imagem regular.
def infer_square_grid_from_vector(vector_size: int, channels: int = 2) -> tuple[int, int] | None:
    if vector_size % channels != 0:
        return None
    pixels_per_channel = vector_size // channels
    side = int(np.sqrt(pixels_per_channel))
    if side * side != pixels_per_channel:
        return None
    return side, side


# Função auxiliar SAE: carrega coordenadas físicas x,y para interpolar vetores OpenFOAM não estruturados em uma grade regular.
def load_reconstruction_coordinates(case_dir: Path, n_cells: int) -> tuple[np.ndarray, np.ndarray]:
    pod_files = (
        case_dir / "POD" / "resultados" / "resultados_pod_completos.npz",
        case_dir / "POD" / "resultados" / "resultados_pod_basicos.npz",
    )
    for pod_file in pod_files:
        if pod_file.exists():
            pod_data = np.load(pod_file, allow_pickle=True)
            if "x" in pod_data.files and "y" in pod_data.files:
                x = np.asarray(pod_data["x"], dtype=np.float64).ravel()
                y = np.asarray(pod_data["y"], dtype=np.float64).ravel()
                if x.size == n_cells and y.size == n_cells:
                    return x, y

    center_files = (
        case_dir / "rom_melhorias" / "cache" / "cell_centers.npy",
        case_dir / "rom" / "cache" / "cell_centers.npy",
    )
    for center_file in center_files:
        if center_file.exists():
            centers = np.asarray(np.load(center_file), dtype=np.float64)
            if centers.shape[0] == n_cells and centers.shape[1] >= 2:
                return centers[:, 0], centers[:, 1]

    raise FileNotFoundError("Não foi possível localizar coordenadas x,y para reconstruir os heatmaps SAE.")


# Função auxiliar SAE: interpola valores de células não estruturadas para uma grade regular 2D.
def interpolate_to_grid(
    x: np.ndarray,
    y: np.ndarray,
    values: np.ndarray,
    grid_x: np.ndarray,
    grid_y: np.ndarray,
) -> np.ndarray:
    from scipy.interpolate import griddata

    points = np.column_stack([x, y])
    field = griddata(points, values, (grid_x, grid_y), method="linear")
    if np.isnan(field).any():
        nearest = griddata(points, values, (grid_x, grid_y), method="nearest")
        field = np.where(np.isnan(field), nearest, field)
    return np.asarray(field, dtype=np.float64)


# Função de reconstrução SAE: carrega o decoder do autoencoder e reconstrói campos físicos a partir dos latentes.
def build_sae_snapshot_reconstructor(dataset: ReducedDataset, grid_size: int = 64):
    import tensorflow as tf

    # O modelo e a normalização normalmente ficam na mesma pasta de latents.npz.
    # Se os latentes vierem de uma subpasta, também testamos a pasta pai.
    results_dir = dataset.source.parent
    if not (results_dir / "sae_dense_best.keras").exists() and (results_dir.parent / "sae_dense_best.keras").exists():
        results_dir = results_dir.parent
    model_path = results_dir / "sae_dense_best.keras"
    if not model_path.exists():
        model_path = results_dir / "sae_dense.keras"
    if not model_path.exists():
        raise FileNotFoundError("Could not find a saved SAE model for reconstruction.")

    norm_path = results_dir / "sae_dense_norm.npz"
    if not norm_path.exists():
        raise FileNotFoundError("Could not find sae_dense_norm.npz for denormalization.")

    autoencoder = tf.keras.models.load_model(model_path, compile=False)
    norm_data = np.load(norm_path, allow_pickle=True)
    mu = np.asarray(norm_data["mu"], dtype=np.float64).reshape(-1)
    sig = np.asarray(norm_data["sig"], dtype=np.float64).reshape(-1)
    # O normalizador informa quantos canais o SAE reconstrói. Neste caso: Ux, Uy, alpha.water, p_rgh, k.
    channel_names = [str(name) for name in np.asarray(norm_data["channel_names"]).ravel()] if "channel_names" in norm_data.files else ["Ux", "Uy"]
    n_channels = len(channel_names)

    # Descobre automaticamente quais camadas pertencem ao decoder: todas as camadas depois de "latent".
    # Isso evita erro quando o modelo não possui camadas BatchNorm como dec_bn_0/dec_bn_1.
    layer_names = [layer.name for layer in autoencoder.layers]
    if "latent" not in layer_names:
        raise ValueError(f"Camada 'latent' não encontrada no modelo. Camadas disponíveis: {layer_names}")
    decoder_layer_names = layer_names[layer_names.index("latent") + 1 :]

    latent_input = tf.keras.Input(shape=(dataset.states.shape[1],), name="latent_input")
    x_dec = latent_input
    for layer_name in decoder_layer_names:
        x_dec = autoencoder.get_layer(layer_name)(x_dec)
    decoder = tf.keras.Model(latent_input, x_dec, name="sae_decoder")

    square_shape = infer_square_grid_from_vector(mu.size, channels=n_channels)
    if square_shape is None:
        # Neste caso o SAE reconstrói um vetor OpenFOAM não estruturado por canais.
        # Para plotar como imagem, interpolamos Ux e Uy para uma grade regular.
        if mu.size % n_channels != 0:
            raise ValueError(f"Tamanho do vetor SAE inválido para {n_channels} canais: {mu.size}")
        n_cells = mu.size // n_channels
        # Sobe na árvore de pastas até encontrar a raiz do caso, identificada por conter POD/ ou rom*/.
        case_dir = next(
            (parent for parent in dataset.source.parents if (parent / "POD").exists() or (parent / "rom_melhorias").exists() or (parent / "rom").exists()),
            dataset.source.parents[2],
        )
        x_coord, y_coord = load_reconstruction_coordinates(case_dir, n_cells)
        xmin, xmax = float(np.min(x_coord)), float(np.max(x_coord))
        ymin, ymax = float(np.min(y_coord)), float(np.max(y_coord))
        grid_x, grid_y = np.meshgrid(
            np.linspace(xmin, xmax, grid_size),
            np.linspace(ymin, ymax, grid_size),
        )
    else:
        ny, nx = square_shape

    # Função interna de reconstrução: reconstrói um lote de estados reduzidos para campos físicos de velocidade.
    def reconstruct_batch(reduced_states: np.ndarray) -> np.ndarray:
        reduced_states = np.atleast_2d(np.asarray(reduced_states, dtype=np.float32))
        decoded_norm = decoder.predict(reduced_states, verbose=0)
        decoded = decoded_norm * sig[None, :] + mu[None, :]
        if square_shape is not None:
            # O SAE pode ter mais que 2 canais; para comparação de velocidade usamos apenas Ux e Uy.
            return decoded.reshape((-1, ny, nx, n_channels))[..., :2].astype(np.float64)

        frames = []
        for vector in decoded:
            # Assume organização em blocos por canal: [canal0(células), canal1(células), ...].
            ux = vector[:n_cells]
            uy = vector[n_cells : 2 * n_cells]
            ux_grid = interpolate_to_grid(x_coord, y_coord, ux, grid_x, grid_y)
            uy_grid = interpolate_to_grid(x_coord, y_coord, uy, grid_x, grid_y)
            frames.append(np.stack([ux_grid, uy_grid], axis=-1))
        return np.asarray(frames, dtype=np.float64)

    return reconstruct_batch


# Função de visualização de campo: salva Ux, Uy e magnitude da velocidade para um único snapshot.
def save_velocity_snapshot(frame: np.ndarray, output_path: Path, title: str) -> None:
    force_matplotlib_agg()
    ux = frame[..., 0]
    uy = frame[..., 1]
    umag = np.sqrt(ux**2 + uy**2)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for axis, values, label in zip(axes, (ux, uy, umag), ("Ux", "Uy", "|U|")):
        image = axis.imshow(values, origin="lower", cmap="viridis")
        axis.set_title(label)
        axis.axis("off")
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de comparação de campo: salva referência, SINDy e erro para Ux, Uy e |U|.
def save_velocity_comparison(
    reference_frame: np.ndarray,
    predicted_frame: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    force_matplotlib_agg()
    ref_ux = reference_frame[..., 0]
    ref_uy = reference_frame[..., 1]
    ref_umag = np.sqrt(ref_ux**2 + ref_uy**2)
    pred_ux = predicted_frame[..., 0]
    pred_uy = predicted_frame[..., 1]
    pred_umag = np.sqrt(pred_ux**2 + pred_uy**2)

    panels = (
        (ref_ux, pred_ux, pred_ux - ref_ux, "Ux"),
        (ref_uy, pred_uy, pred_uy - ref_uy, "Uy"),
        (ref_umag, pred_umag, pred_umag - ref_umag, "|U|"),
    )

    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    for row, (ref_values, pred_values, err_values, label) in enumerate(panels):
        for col, values, suffix in (
            (0, ref_values, "ref"),
            (1, pred_values, "SINDy"),
            (2, err_values, "err"),
        ):
            image = axes[row, col].imshow(values, origin="lower", cmap="viridis")
            axes[row, col].set_title(f"{label} {suffix}")
            axes[row, col].axis("off")
            fig.colorbar(image, ax=axes[row, col], fraction=0.046, pad=0.04)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de exemplos físicos: reconstrói e salva heatmaps nos tempos solicitados pelo usuário.
def generate_requested_time_heatmaps(
    dataset: ReducedDataset,
    trajectory_pred: np.ndarray | None,
    trajectory_method: str | None,
    output_dir: Path,
    config: SindyConfig,
) -> dict[str, Any] | None:
    if not config.heatmap_times:
        return None

    print(f"   Generating heatmaps for requested times: {config.heatmap_times}")
    matched, missing, tolerance = match_requested_times(dataset.times, config.heatmap_times)
    heatmap_dir = output_dir / "heatmaps_exact_times"
    heatmap_dir.mkdir(parents=True, exist_ok=True)

    reconstruct_batch = build_sae_snapshot_reconstructor(dataset, config.heatmap_grid_size)
    if matched:
        indices = [int(item["index"]) for item in matched]
        reference_frames = reconstruct_batch(dataset.states[indices])
        predicted_frames = reconstruct_batch(trajectory_pred[indices]) if trajectory_pred is not None else None

        for position, item in enumerate(matched):
            requested = float(item["requested_time"])
            actual = float(item["actual_time"])
            target_file = heatmap_dir / f"time_{requested:04.1f}s.png"
            if predicted_frames is None:
                title = f"SAE reconstruction | requested={requested:.3f}s | used={actual:.3f}s"
                save_velocity_snapshot(reference_frames[position], target_file, title)
            else:
                field_error = relative_l2(reference_frames[position], predicted_frames[position])
                sindy_label = "SINDy" if trajectory_method is None else f"SINDy ({trajectory_method})"
                title = (
                    f"SAE vs {sindy_label} | requested={requested:.3f}s | "
                    f"used={actual:.3f}s | relL2={field_error:.4f}"
                )
                save_velocity_comparison(
                    reference_frames[position],
                    predicted_frames[position],
                    target_file,
                    title,
                )

    report = {
        "requested_times": [float(value) for value in config.heatmap_times],
        "matched_times": matched,
        "missing_times": [float(value) for value in missing],
        "time_tolerance": float(tolerance),
        "comparison_with_sindy": bool(trajectory_pred is not None),
        "trajectory_method": trajectory_method,
    }
    (heatmap_dir / "report.json").write_text(json.dumps(make_json_ready(report), indent=2), encoding="utf-8")
    if missing:
        print(f"   Missing exact times for SAE: {missing}")
    return report


## 7. Salvamento dos resultados e orquestração do pipeline
Centraliza geração de arquivos, resumo JSON/NPZ e execução ordenada de todas as etapas.


In [ ]:
# Esta célula salva os resultados e organiza a execução completa do pipeline.
# Função de salvamento: grava equações, figuras, NPZ e summary.json com métricas e metadados.
def save_results(
    dataset: ReducedDataset,
    library: PolynomialLibrary,
    best_result: dict[str, Any],
    scan_results: list[dict[str, Any]],
    trajectory_pred: np.ndarray | None,
    trajectory_error: float | None,
    integration_message: str | None,
    trajectory_method: str | None,
    derivative_best_alpha: float,
    output_dir: Path,
    config: SindyConfig,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    equations = format_equations(best_result["Xi"], library.feature_names, dataset.state_symbol)
    (output_dir / "equations.txt").write_text(equations, encoding="utf-8")

    plot_state_heatmap(dataset.times, dataset.states, output_dir, dataset.state_symbol, "SAE reduced coordinates")
    plot_alpha_scan(scan_results, output_dir, title="SAE + SINDy alpha scan")
    plot_derivative_comparison(
        dataset.times,
        best_result["derivatives_reference"],
        best_result["derivatives_pred"],
        output_dir,
        dataset.state_symbol,
        config.max_plot_states,
    )
    plot_state_comparison(
        dataset.times,
        dataset.states,
        trajectory_pred,
        "Reference",
        "SINDy" if trajectory_method is None else f"SINDy ({trajectory_method})",
        output_dir,
        dataset.state_symbol,
        config.max_plot_states,
    )
    heatmap_report = generate_requested_time_heatmaps(
        dataset,
        trajectory_pred,
        trajectory_method,
        output_dir,
        config,
    )

    summary = {
        "architecture": "SAE+SINDy",
        "source_file": str(dataset.source),
        "n_snapshots": int(dataset.states.shape[0]),
        "n_coordinates": int(dataset.states.shape[1]),
        "dt": dataset.metadata.get("dt"),
        "polynomial_order": int(config.polynomial_order),
        "use_sine": bool(config.use_sine),
        "selected_alpha": float(best_result["alpha"]),
        "derivative_best_alpha": float(derivative_best_alpha),
        "library_size": int(best_result["Xi"].shape[0]),
        "derivative_l2_error": float(best_result["derivative_l2_error"]),
        "trajectory_l2_error": None if trajectory_error is None else float(trajectory_error),
        "trajectory_method": trajectory_method,
        "n_nonzero_terms": int(best_result["n_nonzero"]),
        "integration_message": integration_message,
        "metadata": make_json_ready(dataset.metadata),
        "heatmap_examples": make_json_ready(heatmap_report),
    }

    np.savez_compressed(
        output_dir / "sindy_results.npz",
        times=dataset.times,
        states=dataset.states,
        Xi=best_result["Xi"],
        feature_names=np.asarray(library.feature_names, dtype=object),
        alpha=np.array([best_result["alpha"]], dtype=np.float64),
        derivative_l2_error=np.array([best_result["derivative_l2_error"]], dtype=np.float64),
        n_nonzero=np.array([best_result["n_nonzero"]], dtype=np.int64),
        derivatives_reference=best_result["derivatives_reference"],
        derivatives_pred=best_result["derivatives_pred"],
        trajectory_pred=np.full_like(dataset.states, np.nan) if trajectory_pred is None else trajectory_pred,
    )
    (output_dir / "summary.json").write_text(json.dumps(make_json_ready(summary), indent=2), encoding="utf-8")
    return summary


# Função orquestradora SAE: executa em ordem carregamento, derivadas, biblioteca, identificação SINDy, simulação e salvamento.
def run_sae_analysis(
    case_dir: Path,
    config: SindyConfig | None = None,
    output_root: Path | None = None,
) -> dict[str, Any]:
    case_dir = Path(case_dir).resolve()
    output_root = case_dir / "SINDy" / "resultados" if output_root is None else Path(output_root)
    output_dir = output_root / "sae_sindy"
    output_dir.mkdir(parents=True, exist_ok=True)

    if config is None:
        config = SindyConfig()

    # Passo 1: carregar vetores latentes SAE que serão os estados reduzidos.
    dataset = load_sae_dataset(case_dir)

    print("\nProcessing SAE + SINDy")
    print(f"   Source: {dataset.source}")
    print(f"   Reduced coordinates: {dataset.states.shape}")
    print(f"   Time range: [{dataset.times[0]:.6f}, {dataset.times[-1]:.6f}]")

    # Passo 2: calcular derivadas temporais dX/dt a partir dos dados reduzidos.
    derivatives = compute_derivatives(dataset.states, dataset.times)
    # Passo 3: construir a biblioteca candidata Theta(X), com termos polinomiais e seno/cosseno.
    library = PolynomialLibrary(
        n_variables=dataset.states.shape[1],
        order=config.polynomial_order,
        use_sine=config.use_sine,
        variable_symbol=dataset.state_symbol,
    )
    # Passo 4: avaliar a biblioteca nos dados para formar a matriz Theta.
    theta = library.transform(dataset.states)
    # Passo 5: descobrir as equações esparsas, isto é, os coeficientes Xi.
    best_result, scan_results = discover_equations(theta, derivatives, config.alpha_values)
    derivative_best_alpha = float(best_result["alpha"])

    for result in [best_result, *scan_results]:
        result["derivatives_reference"] = derivatives

    # Passo 6: selecionar inicialmente o melhor modelo pelas derivadas e tentar integração livre.
    selected_result = best_result
    trajectory_method = "solve_ivp"
    trajectory_pred, integration_message = simulate_sindy(
        library,
        selected_result["Xi"],
        dataset.times,
        dataset.states[0],
    )

    # Passo 7: se a integração livre falhar, usa previsão de um passo guiada pela referência.
    if trajectory_pred is None:
        print(f"   Integration warning: {integration_message}")
        for result in scan_results:
            result["reference_guided_pred"] = simulate_reference_guided_sindy(
                library,
                result["Xi"],
                dataset.times,
                dataset.states,
            )
            result["reference_guided_l2_error"] = relative_l2(
                dataset.states,
                result["reference_guided_pred"],
            )
        selected_result = min(scan_results, key=lambda item: item["reference_guided_l2_error"])
        selected_result["derivatives_reference"] = derivatives
        trajectory_pred = selected_result["reference_guided_pred"]
        trajectory_error = float(selected_result["reference_guided_l2_error"])
        trajectory_method = "reference_guided_one_step"
        integration_message = f"{integration_message} | Fallback: reference-guided one-step SINDy"
        print(
            f"   Fallback alpha={selected_result['alpha']:.4g} "
            f"with trajectory relative L2={trajectory_error:.6f}"
        )
    else:
        trajectory_error = relative_l2(dataset.states, trajectory_pred)
        print(f"   Trajectory relative L2: {trajectory_error:.6f}")

    # Passo 8: salvar equações, figuras, métricas e arquivos NPZ/JSON.
    summary = save_results(
        dataset,
        library,
        selected_result,
        scan_results,
        trajectory_pred,
        trajectory_error,
        integration_message,
        trajectory_method,
        derivative_best_alpha,
        output_dir,
        config,
    )
    print(f"   Results saved to: {output_dir}")
    return summary


## 8. Executar análise
Execute esta célula para rodar o caso completo com os parâmetros definidos acima.


In [ ]:
# Esta célula executa o pipeline completo com a configuração definida acima.
# Reforça o backend Agg caso o kernel já tenha carregado matplotlib_inline antes.
plt.switch_backend("Agg")
summary = run_sae_analysis(CASE_DIR, config, OUTPUT_ROOT)
summary


## 9. Resumo em JSON


In [ ]:
# Esta célula imprime o resumo final em JSON para inspeção rápida das métricas e arquivos gerados.
print(json.dumps(summary, indent=2, ensure_ascii=False))
